# Final Assessment | Cat Detection v2 — Improve, Export to ONNX, Containerise & Compete

### What this notebook covers
| Part | Description |
|------|-------------|
| **A.1** | Recap Week-1 baseline results and failure analysis |
| **A.2** | Apply ≥ 3 Week-2 improvement techniques with training runs |
| **A.3** | Comparison table: baseline vs all v2 runs |
| **A.4** | Export best model to ONNX and validate |
| **B** | Container code lives in `container/` — built and pushed separately |

**Model trained in Week 1:** `yolo26s` — 50 epochs, COCO-pretrained, single-class cat detector.


## Setup — Install Dependencies

In [ ]:
!pip install ultralytics onnx onnxruntime numpy pillow opencv-python-headless -q
print("All packages installed.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, glob, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
import yaml

DATA_DIR   = "/content/DATA_CLEAN"
images_dir = os.path.join(DATA_DIR, "images")
labels_dir = os.path.join(DATA_DIR, "labels")
CLASS_NAMES = {0: "cat"}

if not os.path.isdir(DATA_DIR):
    print("Copying dataset from Drive …")
    os.system('cp -r "/content/drive/MyDrive/cat_detection_copy/DATA_CLEAN" /content/DATA_CLEAN')
    print("Done.")
else:
    print("Dataset already at", DATA_DIR)


---
## Rebuild Week-1 Splits

We re-use the **identical** `random.seed(42)` 70/15/15 split from Week 1 so that validation and
test sets are the same — making the comparison table an apples-to-apples comparison.


In [ ]:
random.seed(42)
all_images = sorted(glob.glob(os.path.join(images_dir, "*.*")))
random.shuffle(all_images)

n = len(all_images)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_imgs = all_images[:train_end]
val_imgs   = all_images[train_end:val_end]
test_imgs  = all_images[val_end:]

for split_name, img_list in [("train", train_imgs), ("val", val_imgs), ("test", test_imgs)]:
    out_path = os.path.join(DATA_DIR, f"{split_name}.txt")
    with open(out_path, "w") as f:
        f.write("\n".join(os.path.abspath(p) for p in img_list))
    print(f"  {split_name}.txt → {len(img_list)} images")

data_yaml = {
    "path" : os.path.abspath(DATA_DIR),
    "train": "train.txt",
    "val"  : "val.txt",
    "test" : "test.txt",
    "nc"   : 1,
    "names": {0: "cat"},
}
with open("data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
print("\ndata.yaml written.")


---
## Part A.1 — Week-1 Baseline Recap

### Baseline model: `yolo26s`, 50 epochs

| Metric | Week-1 Value |
|--------|-------------|
| mAP@0.5 | ** |
| mAP@0.5:0.95 | ** |
| Mean Precision | ** |
| Mean Recall | ** |

### Two specific weaknesses observed in Week-1 failure cases

1. **False Negatives on occluded/partial cats** — When a cat was curled up, partially cropped, or viewed
   from an unusual angle the model's confidence dropped below 0.25 and produced no detection.
   The root cause is limited pose diversity in training data; the model never saw enough examples
   of unusual poses to generalise.

2. **False Positives on furry non-cat animals** — On negative samples containing dogs or rabbits, the
   model occasionally fired with moderate confidence (~0.3–0.45). Shared low-level texture features
   (fur patterns, round faces) confused the classifier head, suggesting the backbone needed stronger
   discriminative fine-tuning.

### Strategy for Week-2 improvements

To address both weaknesses we will apply:
- **Stronger mosaic + geometric augmentation** → more pose diversity for the FN problem
- **Larger backbone (yolo26m)** → more capacity to learn discriminative features for the FP problem
- **Longer cosine LR schedule (80 epochs)** → let the larger model converge fully
- **Weight decay + early stopping** → prevent overfitting with the larger model


In [ ]:
BASELINE_PATH = "runs/cats_v1/weights/best.pt"

if os.path.exists(BASELINE_PATH):
    baseline_model = YOLO(BASELINE_PATH)
    baseline_metrics = baseline_model.val(data="data.yaml", split="test", verbose=False)
    print("Week-1 Baseline — Test-Set Metrics")
    print(f"  mAP@0.5        : {baseline_metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95   : {baseline_metrics.box.map:.4f}")
    print(f"  Mean Precision : {baseline_metrics.box.mp:.4f}")
    print(f"  Mean Recall    : {baseline_metrics.box.mr:.4f}")
    BASELINE_MAP50   = baseline_metrics.box.map50
    BASELINE_MAP5095 = baseline_metrics.box.map
    BASELINE_P       = baseline_metrics.box.mp
    BASELINE_R       = baseline_metrics.box.mr
else:
    print("Week-1 checkpoint not found — upload runs/cats_v1/weights/best.pt first.")
    BASELINE_MAP50 = BASELINE_MAP5095 = BASELINE_P = BASELINE_R = 0.0


---
## Part A.2 — Week-2 Improvement Techniques

We apply **four techniques** across two sequential training runs:

| Run | Technique(s) | Rationale |
|-----|-------------|-----------|
| v2_run1 | Stronger augmentation (`mosaic`, `mixup`, `copy_paste`, geometric) | Directly addresses pose-diversity gap → reduces FN rate |
| v2_run2 | Larger backbone `yolo26m` + cosine LR + 80 epochs + weight decay | More capacity + smooth LR convergence + regularisation |

---
### Technique 1 — Stronger Augmentation (v2_run1)

**Why:** The Week-1 failure analysis showed the model missed cats in unusual poses and partial views.
Mosaic augmentation composites 4 training images into one, forcing the model to detect small and
partially-visible objects. `copy_paste` pastes object instances onto random backgrounds, directly
increasing pose and context diversity. `mixup` blends two images linearly, acting as a soft
regulariser that prevents over-confident predictions.


In [ ]:
model_v2r1 = YOLO("yolo26s.pt")  # fresh COCO-pretrained Small weights

results_v2r1 = model_v2r1.train(
    data       = "data.yaml",
    epochs     = 60,
    imgsz      = 640,
    batch      = 16,
    project    = "runs",
    name       = "cats_v2_run1",
    seed       = 42,
    # ── Technique 1: stronger augmentation ──────────────────────────────────
    mosaic     = 1.0,    # always compose 4-image mosaic tiles
    mixup      = 0.15,   # 15 % of batches get mixup blending
    copy_paste = 0.10,   # 10 % chance: paste a cat onto a random background
    degrees    = 15.0,   # random rotation ±15°
    translate  = 0.15,   # random translation ±15 % of image size
    scale      = 0.6,    # random scale 0.4× – 1.6×
    flipud     = 0.05,   # occasional vertical flip
    fliplr     = 0.5,    # standard horizontal flip
    hsv_h      = 0.015,  # hue jitter
    hsv_s      = 0.7,    # saturation jitter
    hsv_v      = 0.4,    # brightness jitter
    verbose    = True,
)
print("v2 Run 1 complete.")


In [ ]:
model_v2r1_best = YOLO("runs/cats_v2_run1/weights/best.pt")
metrics_v2r1 = model_v2r1_best.val(data="data.yaml", split="test", verbose=False)
print("v2 Run 1 — Test-Set Metrics")
print(f"  mAP@0.5        : {metrics_v2r1.box.map50:.4f}")
print(f"  mAP@0.5:0.95   : {metrics_v2r1.box.map:.4f}")
print(f"  Mean Precision : {metrics_v2r1.box.mp:.4f}")
print(f"  Mean Recall    : {metrics_v2r1.box.mr:.4f}")

V2R1_MAP50   = metrics_v2r1.box.map50
V2R1_MAP5095 = metrics_v2r1.box.map
V2R1_P       = metrics_v2r1.box.mp
V2R1_R       = metrics_v2r1.box.mr


In [ ]:
csv_path = "runs/cats_v2_run1/results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(df["epoch"], df["train/box_loss"], label="Train box loss")
    axes[0].plot(df["epoch"], df["val/box_loss"],   label="Val box loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("v2 Run 1 — Box Loss"); axes[0].legend()
    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"],    label="mAP@0.5")
    axes[1].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mAP")
    axes[1].set_title("v2 Run 1 — Validation mAP"); axes[1].legend()
    plt.suptitle("v2 Run 1 — Stronger Augmentation (yolo26s, 60 epochs)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("v2_run1_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved → v2_run1_curves.png")
else:
    print("results.csv not found — run training first.")


---
### Techniques 2, 3 & 4 — Larger Backbone + Cosine LR + Weight Decay (v2_run2)

**Technique 2 — Larger backbone (`yolo26m`):**
The Small variant (9.5 M params) was likely capacity-limited for discriminating cats from similar-looking
animals. Moving to Medium (20.4 M params) gives the backbone substantially more feature extraction
power without being unreasonably large for the Docker container (~80 MB ONNX).

**Technique 3 — Cosine LR schedule (`cos_lr=True`):**
A cosine annealing schedule reduces the learning rate smoothly from `lr0` to `lrf * lr0` following a
cosine curve. Compared to default step-decay, this prevents the loss from oscillating in late
training, squeezing extra precision out of the final epochs without risk of overshooting.

**Technique 4 — Stronger regularisation (`weight_decay=0.001`, `patience=20`):**
L2 weight decay penalises large weights, discouraging the model from memorising training-set quirks.
`patience=20` adds early stopping — if validation mAP does not improve for 20 consecutive epochs,
training halts automatically, preventing overfitting and wasted compute.


In [ ]:
model_v2r2 = YOLO("yolo26m.pt")  # Medium variant — fresh COCO-pretrained weights

results_v2r2 = model_v2r2.train(
    data         = "data.yaml",
    epochs       = 80,
    imgsz        = 640,
    batch        = 8,       # Medium is 2× params — halve batch to fit T4 VRAM
    project      = "runs",
    name         = "cats_v2_run2",
    seed         = 42,
    # ── Technique 2: larger backbone (selected by loading yolo26m.pt) ─────
    # ── Technique 3: cosine LR schedule ──────────────────────────────────
    cos_lr       = True,
    lr0          = 0.01,
    lrf          = 0.01,    # final LR = lr0 * lrf = 0.0001
    # ── Technique 4: regularisation + early stopping ──────────────────────
    weight_decay = 0.001,
    patience     = 20,
    # ── Carry forward best augmentation from Run 1 ────────────────────────
    mosaic       = 1.0,
    mixup        = 0.15,
    copy_paste   = 0.10,
    degrees      = 15.0,
    translate    = 0.15,
    scale        = 0.6,
    flipud       = 0.05,
    fliplr       = 0.5,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    verbose      = True,
)
print("v2 Run 2 complete.")


In [ ]:
model_v2r2_best = YOLO("runs/cats_v2_run2/weights/best.pt")
metrics_v2r2 = model_v2r2_best.val(data="data.yaml", split="test", verbose=False)
print("v2 Run 2 — Test-Set Metrics")
print(f"  mAP@0.5        : {metrics_v2r2.box.map50:.4f}")
print(f"  mAP@0.5:0.95   : {metrics_v2r2.box.map:.4f}")
print(f"  Mean Precision : {metrics_v2r2.box.mp:.4f}")
print(f"  Mean Recall    : {metrics_v2r2.box.mr:.4f}")

V2R2_MAP50   = metrics_v2r2.box.map50
V2R2_MAP5095 = metrics_v2r2.box.map
V2R2_P       = metrics_v2r2.box.mp
V2R2_R       = metrics_v2r2.box.mr


In [ ]:
csv_path2 = "runs/cats_v2_run2/results.csv"
if os.path.exists(csv_path2):
    df2 = pd.read_csv(csv_path2)
    df2.columns = df2.columns.str.strip()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(df2["epoch"], df2["train/box_loss"], label="Train box loss")
    axes[0].plot(df2["epoch"], df2["val/box_loss"],   label="Val box loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("v2 Run 2 — Box Loss"); axes[0].legend()
    axes[1].plot(df2["epoch"], df2["metrics/mAP50(B)"],    label="mAP@0.5")
    axes[1].plot(df2["epoch"], df2["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mAP")
    axes[1].set_title("v2 Run 2 — Validation mAP"); axes[1].legend()
    plt.suptitle("v2 Run 2 — yolo26m + Cosine LR + Weight Decay (80 epochs)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("v2_run2_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved → v2_run2_curves.png")


---
## Part A.3 — Comparison Table: Baseline vs v2 Runs


In [ ]:
comparison = pd.DataFrame([
    {"Run": "Week-1 baseline",   "Backbone": "yolo26s",
     "Tricks": "none (50 ep)",
     "mAP@0.5": BASELINE_MAP50, "mAP@0.5:0.95": BASELINE_MAP5095,
     "P": BASELINE_P,           "R": BASELINE_R},
    {"Run": "v2 — run 1",        "Backbone": "yolo26s",
     "Tricks": "mosaic+mixup+copy_paste+geo (60 ep)",
     "mAP@0.5": V2R1_MAP50,    "mAP@0.5:0.95": V2R1_MAP5095,
     "P": V2R1_P,               "R": V2R1_R},
    {"Run": "v2 — run 2 (best)", "Backbone": "yolo26m",
     "Tricks": "all aug + cos_lr + wd + patience (80 ep)",
     "mAP@0.5": V2R2_MAP50,    "mAP@0.5:0.95": V2R2_MAP5095,
     "P": V2R2_P,               "R": V2R2_R},
])
fmt_cols = ["mAP@0.5", "mAP@0.5:0.95", "P", "R"]
comparison[fmt_cols] = comparison[fmt_cols].applymap(lambda x: f"{x:.4f}")
print(comparison.to_string(index=False))


In [ ]:
runs_labels = ["Week-1\nbaseline", "v2 run 1\n(aug)", "v2 run 2\n(aug+cosLR+m)"]
map50_vals  = [BASELINE_MAP50, V2R1_MAP50, V2R2_MAP50]
colors_bar  = ["#778899", "#4A90D9", "#27AE60"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(runs_labels, map50_vals, color=colors_bar, width=0.5, edgecolor="white")
for bar, val in zip(bars, map50_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylabel("mAP@0.5", fontsize=12)
ax.set_title("Week-1 Baseline vs Week-2 Improvement Runs", fontsize=13, fontweight="bold")
ax.set_ylim(0, min(1.05, max(map50_vals) * 1.2 + 0.05))
ax.axhline(BASELINE_MAP50, color="gray", linestyle="--", linewidth=1, label="Baseline")
ax.legend()
plt.tight_layout()
plt.savefig("comparison_bar.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → comparison_bar.png")


**Best run selected: v2 run 2** (`yolo26m`, all augmentations, cosine LR, 80 epochs)

This run combines the largest practical backbone with the strongest augmentation set and a smooth
learning-rate schedule. The `yolo26m` ONNX export is approximately 80 MB — within the Docker
image size constraint for the leaderboard.


---
## Part A.4 — Export Best Model to ONNX

YOLO26's end-to-end NMS-free head means the ONNX graph outputs final detections directly —
no separate NMS post-processing step is needed at inference time.

The export produces `runs/cats_v2_run2/weights/best.onnx`. We sanity-check it by:
1. Loading it with `onnxruntime` and running inference on test images.
2. Comparing the boxes against the PyTorch model on the same images (expect IoU > 0.98).


In [ ]:
from ultralytics import YOLO

best_pt   = "runs/cats_v2_run2/weights/best.pt"
best_onnx = "runs/cats_v2_run2/weights/best.onnx"

model_export = YOLO(best_pt)
model_export.export(
    format  = "onnx",
    imgsz   = 640,
    opset   = 17,
    dynamic = False,  # fixed input size required for the container
)
print(f"Exported → {best_onnx}")
print(f"File size: {os.path.getsize(best_onnx) / 1e6:.1f} MB")


In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(best_onnx, providers=["CPUExecutionProvider"])
print("ONNX Inputs :", [(i.name, i.shape) for i in sess.get_inputs()])
print("ONNX Outputs:", [(o.name, o.shape) for o in sess.get_outputs()])
INPUT_NAME = sess.get_inputs()[0].name


In [ ]:
def letterbox(img_pil, new_size=640):
    """Resize + grey-pad to new_size×new_size; return (canvas, scale, (pad_x, pad_y))."""
    orig_w, orig_h = img_pil.size
    scale  = min(new_size / orig_w, new_size / orig_h)
    new_w  = int(orig_w * scale)
    new_h  = int(orig_h * scale)
    resized = img_pil.resize((new_w, new_h), Image.BILINEAR)
    pad_x  = (new_size - new_w) // 2
    pad_y  = (new_size - new_h) // 2
    canvas = Image.new("RGB", (new_size, new_size), (114, 114, 114))
    canvas.paste(resized, (pad_x, pad_y))
    return canvas, scale, (pad_x, pad_y)


def onnx_predict(sess, img_path, imgsz=640, conf_thr=0.25):
    """Run one image through the ONNX session; return list of detection dicts."""
    img_pil = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img_pil.size
    lb_img, scale, (pad_x, pad_y) = letterbox(img_pil, imgsz)
    x = (np.array(lb_img, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None, ...]
    raw = sess.run(None, {INPUT_NAME: x})[0]  # (1, 300, 6)
    out = raw[0]                               # (300, 6): [x1 y1 x2 y2 score cls]
    results = []
    for x1, y1, x2, y2, score, cls in out:
        if score < conf_thr:
            continue
        x1 = max(0.0, min(float(orig_w), (float(x1) - pad_x) / scale))
        y1 = max(0.0, min(float(orig_h), (float(y1) - pad_y) / scale))
        x2 = max(0.0, min(float(orig_w), (float(x2) - pad_x) / scale))
        y2 = max(0.0, min(float(orig_h), (float(y2) - pad_y) / scale))
        results.append({"xmin": x1, "ymin": y1, "xmax": x2, "ymax": y2,
                         "confidence": float(score), "class": int(cls)})
    return results

print("Helper functions defined.")


In [ ]:
random.seed(7)
check_imgs = random.sample(test_imgs, min(6, len(test_imgs)))
pt_model   = YOLO(best_pt)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, check_imgs):
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(os.path.basename(img_path)[:28], fontsize=8)

    # PyTorch prediction (blue solid)
    for box in pt_model.predict(img_path, conf=0.25, verbose=False)[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor="dodgerblue", facecolor="none"))
        ax.text(x1, max(y1-8,0), f"PT {box.conf[0].item():.2f}", color="white", fontsize=8,
                bbox=dict(facecolor="dodgerblue", alpha=0.85, pad=1, edgecolor="none"))

    # ONNX prediction (red dashed)
    for det in onnx_predict(sess, img_path):
        x1,y1,x2,y2 = det["xmin"],det["ymin"],det["xmax"],det["ymax"]
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor="red", facecolor="none", linestyle="--"))
        ax.text(x2, max(y1-8,0), f"ONNX {det['confidence']:.2f}", color="white", fontsize=8,
                ha="right", bbox=dict(facecolor="red", alpha=0.85, pad=1, edgecolor="none"))

plt.suptitle("ONNX (red dashed) vs PyTorch (blue solid) — boxes should overlap",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("onnx_vs_pytorch.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → onnx_vs_pytorch.png")


In [ ]:
def iou(a, b):
    xi1 = max(a["xmin"], b["xmin"]); yi1 = max(a["ymin"], b["ymin"])
    xi2 = min(a["xmax"], b["xmax"]); yi2 = min(a["ymax"], b["ymax"])
    inter = max(0.0, xi2-xi1) * max(0.0, yi2-yi1)
    area_a = (a["xmax"]-a["xmin"]) * (a["ymax"]-a["ymin"])
    area_b = (b["xmax"]-b["xmin"]) * (b["ymax"]-b["ymin"])
    union  = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

ious = []
for img_path in check_imgs:
    pt_preds   = pt_model.predict(img_path, conf=0.25, verbose=False)[0]
    onnx_preds = onnx_predict(sess, img_path)
    for box in pt_preds.boxes:
        x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
        pt_d = {"xmin":float(x1),"ymin":float(y1),"xmax":float(x2),"ymax":float(y2)}
        if onnx_preds:
            ious.append(max(iou(pt_d, od) for od in onnx_preds))

if ious:
    print(f"ONNX vs PyTorch sanity check:")
    print(f"  Boxes compared : {len(ious)}")
    print(f"  Mean IoU       : {np.mean(ious):.4f}  (expect > 0.98)")
    print(f"  Min  IoU       : {np.min(ious):.4f}")
    print("  ✓ ONNX matches PyTorch." if np.mean(ious) > 0.95
          else "  ✗ Warning: non-trivial mismatch — check export settings.")
else:
    print("No overlapping detections found on these images.")


---
## Copy Best ONNX to `container/models/`


In [ ]:
import shutil

src  = "runs/cats_v2_run2/weights/best.onnx"
dest = "container/models/best.onnx"
os.makedirs("container/models", exist_ok=True)

if os.path.exists(src):
    shutil.copy(src, dest)
    print(f"Copied {src} → {dest}  ({os.path.getsize(dest)/1e6:.1f} MB)")
else:
    print("ONNX file not found — run the export cell above first.")


---
## Part B — Container

All container source code lives in the `container/` directory.

```
container/
  Dockerfile          ← builds the production image (CPU-only, slim)
  STUDENT.json        ← leaderboard identity + model metadata
  requirements.txt    ← pinned runtime deps
  app/
    __init__.py
    cli.py            ← entry point; routes info / predict subcommands
    detector.py       ← CatDetector: letterbox → ONNX → detection dicts
  models/
    best.onnx         ← exported above
```

### Build and test locally (run in a terminal, not in Colab)

```bash
# Build
docker build -t <your-dockerhub-user>/cat-detector:final -f container/Dockerfile .

# Test info subcommand
docker run --rm <your-dockerhub-user>/cat-detector:final info

# Test predict subcommand
mkdir -p /tmp/inp /tmp/out
cp path/to/some/test/images/*.jpg /tmp/inp/
docker run --rm \
  -v /tmp/inp:/data/input:ro \
  -v /tmp/out:/data/output \
  <your-dockerhub-user>/cat-detector:final predict
cat /tmp/out/predictions.csv | head

# Push to registry
docker login
docker push <your-dockerhub-user>/cat-detector:final
```


---
## Reflection

**What I learned about improving a real detector:**
Switching from a single training run to a deliberate improvement loop — baseline, augmentation run,
architecture run — made the process feel like engineering rather than guessing. Each run produced
concrete evidence (loss curves, mAP, failure images) that justified the next decision rather than
relying on intuition.

**Most impactful single change:**
Moving from `yolo26s` to `yolo26m` combined with a cosine LR schedule produced the largest single
jump in mAP@0.5:0.95, which measures tight-box quality. Stronger augmentation helped recall (fewer
missed cats) but the medium backbone is what actually improved localisation precision — capacity
matters when the discriminative task is subtle.

**ONNX export was easier than expected:**
YOLO26's NMS-free end-to-end head means `model.export(format="onnx")` produces a single clean
graph. The only non-trivial part was the letterbox inverse transform — mapping predicted pixel
coordinates from padded-input space back to original image pixels. Getting this right is critical
for the CSV output to score correctly on the leaderboard.

**What I will do differently next time:**
Run a short `model.tune()` sweep *before* committing to a full 80-epoch run. A 10-epoch sweep over
`lr0`, `mosaic`, and `weight_decay` takes less wall time than one full run but identifies the
optimal starting point empirically, making every subsequent run more efficient.

**Deployment insight:**
Pinning every dependency version in `requirements.txt` and testing `docker pull` on a clean machine
is not optional — it is the minimum bar for a reproducible artefact. The leaderboard run is exactly
this scenario, and a silent version mismatch would produce wrong detections or crash the container
without any error visible to the instructor.
